# 5 — TF-IDF + Logistic Regression (first trained model)

**Objective:** Train the baseline multi-label symptom → condition classifier using the **frozen training split**, select a classification threshold using **validation only**, and save the fitted model and validation metrics.

This notebook:
- Reads `train.json` and `validation.json` created by `4_prepare_training_data.ipynb`.
- Uses `symptoms_classical` as input and **all** entries of `labels` as targets.
- Trains one-vs-rest Logistic Regression on TF-IDF word n-grams.
- Reports validation Top-1/3/5 *hit rate*, label Recall@K, precision, recall, and F1.
- Writes a reusable local model bundle and JSON validation report.

**The test set is intentionally not loaded here.** Keep it untouched until the final comparison notebook. The generated examples are NHS-source-derived symptom subsets, **not independent patient cases**. The scores are for an educational prototype, not clinical diagnosis.

In [ ]:
from __future__ import annotations

import hashlib
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

RANDOM_SEED = 42
MODEL_NAME = "tfidf_logistic_regression_v1"

def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "app").is_dir() and (path / "app" / "dataset").is_dir():
            return path
    raise FileNotFoundError(
        "Project root not found. Launch the notebook from anywhere inside "
        "the medical-diagnosis repository (the folder containing 'app/dataset')."
    )

ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / "app" / "dataset" / "output" / "training"
TRAIN_FILE = DATA_DIR / "train.json"
VAL_FILE = DATA_DIR / "validation.json"
MODEL_FILE = DATA_DIR / f"{MODEL_NAME}.joblib"
METRICS_FILE = DATA_DIR / f"{MODEL_NAME}_validation_metrics.json"

print("Project root:", ROOT)
print("Train file:", TRAIN_FILE)
print("Validation file:", VAL_FILE)
print("Test set: intentionally not accessed")

## 1. Load and validate the frozen splits

Every record should contain a nonempty cleaned symptom text and a list of one or more condition labels. The same normalized input must not appear in both splits.

**Do not regenerate or resplit data in this notebook.**

In [ ]:
def load_examples(path: Path) -> list[dict]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}. Run 4_prepare_training_data.ipynb first."
        )
    rows = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(rows, list) or not rows:
        raise ValueError(f"Expected a nonempty JSON list: {path}")
    seen_ids = set()
    for index, row in enumerate(rows):
        if not isinstance(row, dict):
            raise ValueError(f"{path.name}: row {index} is not a JSON object")
        text = row.get("symptoms_classical")
        labels = row.get("labels")
        example_id = row.get("example_id")
        if not isinstance(text, str) or not text.strip():
            raise ValueError(f"{path.name}: row {index} has empty symptoms_classical")
        if not isinstance(labels, list) or not labels or not all(
            isinstance(label, str) and label.strip() for label in labels
        ):
            raise ValueError(f"{path.name}: row {index} has invalid labels")
        if not isinstance(example_id, str) or not example_id:
            raise ValueError(f"{path.name}: row {index} has no example_id")
        if example_id in seen_ids:
            raise ValueError(f"{path.name}: duplicate example_id {example_id}")
        seen_ids.add(example_id)
    return rows

train_rows = load_examples(TRAIN_FILE)
val_rows = load_examples(VAL_FILE)

train_texts = [row["symptoms_classical"] for row in train_rows]
val_texts = [row["symptoms_classical"] for row in val_rows]
train_labels = [row["labels"] for row in train_rows]
val_labels = [row["labels"] for row in val_rows]

train_text_set = set(train_texts)
val_text_set = set(val_texts)
assert len(train_text_set) == len(train_rows), "Repeated training inputs found"
assert len(val_text_set) == len(val_rows), "Repeated validation inputs found"
assert not (train_text_set & val_text_set), "Train/validation input leakage!"
assert not ({r["example_id"] for r in train_rows} &
            {r["example_id"] for r in val_rows}), "Repeated example IDs across splits!"

label_binarizer = MultiLabelBinarizer()
y_train = label_binarizer.fit_transform(train_labels)
unknown_validation_labels = sorted(
    set(label for labels in val_labels for label in labels)
    - set(label_binarizer.classes_)
)
assert not unknown_validation_labels, (
    f"Some validation labels are absent from training: {unknown_validation_labels}"
)
y_val = label_binarizer.transform(val_labels)
class_names = label_binarizer.classes_

print(f"Training examples:   {len(train_rows):,}")
print(f"Validation examples: {len(val_rows):,}")
print(f"Training classes:    {len(class_names):,}")
print(f"Validation classes:  {int((y_val.sum(axis=0) > 0).sum()):,}")
print(f"Multi-label training inputs: {sum(len(labels) > 1 for labels in train_labels):,}")
print("Exact train/validation overlap: 0")

## 2. TF-IDF features and One-vs-Rest Logistic Regression

- TF-IDF uses word unigrams and bigrams to capture symptom vocabulary and short phrases.
- Each condition gets its own binary Logistic Regression classifier because one symptom description may match **multiple conditions**.
- We fit the vectorizer **only on training text** to avoid validation leakage.
- Logistic scores are per-label estimates and are **not calibrated medical probabilities**.

In [ ]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=1,
    max_features=50_000,
    sublinear_tf=True,
    dtype=np.float32,
)
X_train = vectorizer.fit_transform(train_texts)
X_val = vectorizer.transform(val_texts)

print("Train TF-IDF shape:", X_train.shape)
print("Validation TF-IDF shape:", X_val.shape)
print("Target matrix shape:", y_train.shape)

In [ ]:
base_model = LogisticRegression(
    solver="liblinear",
    C=2.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
classifier = OneVsRestClassifier(base_model, n_jobs=1)

started = time.perf_counter()
classifier.fit(X_train, y_train)
training_seconds = time.perf_counter() - started

print(f"Training completed in {training_seconds:.1f} seconds")
print(f"Trained {len(class_names)} binary condition classifiers")

## 3. Validate probability ranking (Top-K)

A **Top-K hit** means *at least one* true condition appears among the K highest-ranked labels (since some records have more than one true label). **Recall@K** measures the fraction of *all* true labels retrieved among the Top K, averaged across examples.

These two metrics answer different questions, so both are reported.

In [ ]:
val_scores = np.asarray(classifier.predict_proba(X_val), dtype=np.float64)
assert val_scores.shape == y_val.shape
assert np.isfinite(val_scores).all()
assert (val_scores >= 0).all() and (val_scores <= 1).all()

def ranking_metrics(true_binary: np.ndarray, scores: np.ndarray, k: int) -> dict:
    k = min(k, scores.shape[1])
    top_indices = np.argsort(-scores, axis=1, kind="stable")[:, :k]
    top_true = np.take_along_axis(true_binary, top_indices, axis=1)
    hits = (top_true.sum(axis=1) > 0)
    counts = true_binary.sum(axis=1)
    assert (counts > 0).all()
    return {
        "top_k": int(k),
        "hit_rate": float(hits.mean()),
        "label_recall_at_k": float(np.mean(top_true.sum(axis=1) / counts)),
    }

top_k_results = {f"top_{k}": ranking_metrics(y_val, val_scores, k)
                 for k in (1, 3, 5)}
print(pd.DataFrame(top_k_results).T.to_string())

## 4. Choose the multilabel threshold using validation only

For each condition, a binary prediction is made when its score is at least the selected threshold. We try a small preset grid and select the threshold maximizing **validation micro-F1**. A threshold is for determining the set of predicted labels; it does **not** change the Top-K ranking.

Do not use the held-out test set for threshold selection.

In [ ]:
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60]
threshold_rows = []
for threshold in THRESHOLDS:
    binary_pred = (val_scores >= threshold).astype(np.uint8)
    threshold_rows.append({
        "threshold": float(threshold),
        "micro_f1": float(f1_score(y_val, binary_pred, average="micro", zero_division=0)),
        "micro_precision": float(precision_score(
            y_val, binary_pred, average="micro", zero_division=0)),
        "micro_recall": float(recall_score(
            y_val, binary_pred, average="micro", zero_division=0)),
    })

threshold_table = pd.DataFrame(threshold_rows)
best_row = max(
    threshold_rows,
    key=lambda item: (item["micro_f1"], -item["threshold"])
)
BEST_THRESHOLD = best_row["threshold"]
val_binary = (val_scores >= BEST_THRESHOLD).astype(np.uint8)

supported = np.asarray(y_val.sum(axis=0) > 0).ravel()
validation_metrics = {
    "selected_threshold": float(BEST_THRESHOLD),
    "micro_precision": float(precision_score(
        y_val, val_binary, average="micro", zero_division=0)),
    "micro_recall": float(recall_score(
        y_val, val_binary, average="micro", zero_division=0)),
    "micro_f1": float(f1_score(
        y_val, val_binary, average="micro", zero_division=0)),
    "macro_f1_all_classes": float(f1_score(
        y_val, val_binary, average="macro", zero_division=0)),
    "macro_f1_validation_supported_classes": float(f1_score(
        y_val[:, supported], val_binary[:, supported],
        average="macro", zero_division=0)),
    "weighted_f1": float(f1_score(
        y_val, val_binary, average="weighted", zero_division=0)),
    "top_k": top_k_results,
}
print("Threshold comparison:")
print(threshold_table.round(4).to_string(index=False))
print("\nSelected threshold:", BEST_THRESHOLD)
print("\nFinal validation metrics:")
print(json.dumps(validation_metrics, indent=2))

## 5. Visualize validation Top-K hit rates

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ks = [1, 3, 5]
hit_rates = [100 * top_k_results[f"top_{k}"]["hit_rate"] for k in ks]
bars = ax.bar([f"Top-{k}" for k in ks], hit_rates)
ax.set_ylim(0, 110)
ax.set_ylabel("Validation hit rate (%)")
ax.set_title("TF-IDF + Logistic Regression — validation Top-K")
for bar, value in zip(bars, hit_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 1,
            f"{value:.1f}%", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## 6. Try a free-text example

The list is a ranking of learned **model scores**. In a multi-label classifier, values do **not** sum to 100%, and they are not calibrated estimates of a patient's actual chance of having a disease. We will retrieve warnings and recommendations from MongoDB in the API stage.

In [ ]:
def rank_conditions(symptom_text: str, top_k: int = 5) -> pd.DataFrame:
    if not symptom_text or not symptom_text.strip():
        raise ValueError("Please enter one or more symptoms")
    if top_k < 1:
        raise ValueError("top_k must be positive")
    features = vectorizer.transform([symptom_text])
    scores = classifier.predict_proba(features)[0]
    indices = np.argsort(-scores, kind="stable")[:top_k]
    return pd.DataFrame({
        "condition": class_names[indices],
        "model_score": np.round(scores[indices], 4),
    })

rank_conditions("persistent cough, wheezing, shortness of breath", top_k=5)

## 7. Save the model and validation report

The bundle contains the TF-IDF vocabulary, all Logistic Regression estimators, the ordered class labels, and the validation-selected threshold. Save it locally first; later our **Flask Model API** will upload it to MongoDB GridFS and record metadata in the `models` collection.

Model provenance includes file hashes so the exact frozen training/validation versions are identifiable.

In [ ]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

created_at = datetime.now(timezone.utc).isoformat()
dataset_hashes = {
    "train_json_sha256": file_sha256(TRAIN_FILE),
    "validation_json_sha256": file_sha256(VAL_FILE),
}
metadata = {
    "name": MODEL_NAME,
    "type": "TF-IDF + Logistic Regression (One-vs-Rest multi-label)",
    "createdAt": created_at,
    "training_examples": len(train_rows),
    "validation_examples": len(val_rows),
    "class_count": len(class_names),
    "validation_supported_class_count": int(supported.sum()),
    "validation_missing_class_count": int((~supported).sum()),
    "training_seconds": round(training_seconds, 3),
    "scikit_learn_version": sklearn.__version__,
    "random_seed": RANDOM_SEED,
    "features": "symptoms_classical",
    "tfidf": {
        "ngram_range": [1, 2],
        "max_features": 50_000,
        "sublinear_tf": True,
    },
    "classifier": {
        "solver": "liblinear",
        "C": 2.0,
        "class_weight": "balanced",
    },
    "dataset_sha256": dataset_hashes,
    "evaluation_scope": (
        "NHS source-derived symptom subsets; validation only, not independent "
        "clinical evaluation. Test set not used."
    ),
    "probability_note": (
        "Per-label model scores are not calibrated clinical disease probabilities."
    ),
}
bundle = {
    "vectorizer": vectorizer,
    "classifier": classifier,
    "label_binarizer": label_binarizer,
    "threshold": BEST_THRESHOLD,
    "metadata": metadata,
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(bundle, MODEL_FILE, compress=3)

report = {
    "model": MODEL_NAME,
    "metadata": metadata,
    "validation_metrics": validation_metrics,
    "threshold_search": threshold_rows,
}
METRICS_FILE.write_text(json.dumps(report, ensure_ascii=False, indent=2),
                        encoding="utf-8")

# Verify the persisted bundle loads and reproduces the same top prediction.
reloaded = joblib.load(MODEL_FILE)
sample_text = "persistent cough and wheezing"
sample_features = reloaded["vectorizer"].transform([sample_text])
sample_scores = reloaded["classifier"].predict_proba(sample_features)[0]
top_idx = int(np.argmax(sample_scores))
assert reloaded["label_binarizer"].classes_[top_idx] == class_names[top_idx]

print("Saved model:", MODEL_FILE)
print("Saved validation metrics:", METRICS_FILE)
print(f"Model size: {MODEL_FILE.stat().st_size / (1024 * 1024):.2f} MB")
print("Reload check: passed")
print("\nNEXT: train RNN and LSTM using the SAME frozen train/validation/test splits.")